In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier


PROJECT_ROOT = r"C:\Users\aakif\Documents\DataCompetition"
TRAIN_PATH = PROJECT_ROOT + r"\data\train.csv"

TARGET = "Will_Buy_EV"
ID_COL = "id"

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


# ============================================================
# EXACT 23B IDENTITY ENCODING
# ============================================================

def make_identity_key(series):
    return series.astype("string").fillna("__MISSING__")


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        "value": values,
        "target": target.to_numpy()
    })

    global_mean = float(target.mean())

    stats = (
        temp.groupby("value", dropna=False)["target"]
        .agg(["mean", "count"])
    )

    smoothed = (
        stats["count"] * stats["mean"]
        + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return (
        values.map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def add_identity_features(
    X_fit,
    y_fit,
    X_apply,
    columns,
    n_splits=3,
    smoothing=20
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:

        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(
            len(X_fit),
            dtype=float
        )

        for train_idx, fold_idx in skf.split(
            X_fit,
            y_fit
        ):

            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = (
                apply_mapping(
                    fit_keys.iloc[fold_idx],
                    mapping,
                    global_mean
                ).to_numpy()
            )

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[
            f"{col}__identity_target"
        ] = oof_values

        X_apply[
            f"{col}__identity_target"
        ] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        frequencies = fit_keys.value_counts(
            dropna=False
        )

        X_fit[
            f"{col}__identity_frequency"
        ] = (
            fit_keys
            .map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

        X_apply[
            f"{col}__identity_frequency"
        ] = (
            apply_keys
            .map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

    return X_fit, X_apply


# ============================================================
# DIGIT DECOMPOSITION
# ============================================================

def add_digit_features(X_frame, columns):
    X_frame = X_frame.copy()

    for col in columns:

        values = pd.to_numeric(
            X_frame[col],
            errors="coerce"
        )

        # Rounded absolute integer representation.
        integer_values = (
            values
            .abs()
            .round()
        )

        # Basic magnitude.
        X_frame[
            f"{col}__digits"
        ] = np.floor(
            np.log10(
                integer_values.clip(lower=1)
            )
        ) + 1

        # First digit.
        divisor = (
            10 ** (
                X_frame[f"{col}__digits"] - 1
            )
        )

        X_frame[
            f"{col}__first_digit"
        ] = (
            integer_values / divisor
        ).fillna(0).astype(float)

        X_frame[
            f"{col}__first_digit"
        ] = np.floor(
            X_frame[f"{col}__first_digit"]
        )

        # Last digit.
        X_frame[
            f"{col}__last_digit"
        ] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 10
        )

        # Digit sum.
        def digit_sum(v):
            if pd.isna(v):
                return np.nan

            s = str(int(abs(v)))

            return sum(
                int(ch)
                for ch in s
            )

        X_frame[
            f"{col}__digit_sum"
        ] = integer_values.map(digit_sum)

        # Parity.
        X_frame[
            f"{col}__parity"
        ] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 2
        )

        # Last two digits.
        X_frame[
            f"{col}__mod100"
        ] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 100
        )

        # Last three digits.
        X_frame[
            f"{col}__mod1000"
        ] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 1000
        )

        # Whether value ends in zero.
        X_frame[
            f"{col}__ends_zero"
        ] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 10 == 0
        ).astype(np.int8)

    return X_frame


# ============================================================
# XGBOOST
# ============================================================

def build_preprocessor(X_frame):

    numeric = X_frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical = X_frame.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])

    return ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric
        ),
        (
            "cat",
            categorical_pipeline,
            categorical
        )
    ])


def run_xgb(
    X_tr,
    y_tr,
    X_va,
    y_va,
    label
):

    print("")
    print("------------------------------------------------------------")
    print(label)
    print("------------------------------------------------------------")

    preprocessor = build_preprocessor(
        X_tr
    )

    print("Encoding...")

    X_tr_encoded = preprocessor.fit_transform(
        X_tr
    )

    X_va_encoded = preprocessor.transform(
        X_va
    )

    print(
        "Encoded shape:",
        X_tr_encoded.shape
    )

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        eval_set=[
            (X_va_encoded, y_va)
        ],
        verbose=False
    )

    predictions = model.predict_proba(
        X_va_encoded
    )[:, 1]

    score = roc_auc_score(
        y_va,
        predictions
    )

    print(
        f"{label} ROC-AUC: {score:.6f}"
    )

    return score


# ============================================================
# 33A
# EXACT 23B CONTROL
# ============================================================

print("")
print("============================================================")
print("33A - EXACT 23B CONTROL")
print("============================================================")

X_base_train, X_base_valid = (
    add_identity_features(
        X_train,
        y_train,
        X_valid,
        numeric_cols,
        n_splits=3,
        smoothing=20
    )
)

print(
    "Control feature count:",
    X_base_train.shape[1]
)

score_control = run_xgb(
    X_base_train,
    y_train,
    X_base_valid,
    y_valid,
    "33A_23B_Control"
)

print("")
print(
    f"Control score: {score_control:.6f}"
)
print(
    "Expected benchmark: 0.945243"
)


# ============================================================
# 33B
# IDENTITY + DIGIT DECOMPOSITION
# ============================================================

print("")
print("============================================================")
print("33B - IDENTITY + DIGIT DECOMPOSITION")
print("============================================================")

X_digit_train = add_digit_features(
    X_base_train,
    numeric_cols
)

X_digit_valid = add_digit_features(
    X_base_valid,
    numeric_cols
)

print(
    "Digit attack feature count:",
    X_digit_train.shape[1]
)

score_digit = run_xgb(
    X_digit_train,
    y_train,
    X_digit_valid,
    y_valid,
    "33B_Identity_Frequency_Digits"
)


# ============================================================
# RESULTS
# ============================================================

results = pd.DataFrame([
    {
        "Experiment": "33A_23B_Control",
        "ROC_AUC": score_control
    },
    {
        "Experiment": "33B_Identity_Frequency_Digits",
        "ROC_AUC": score_digit
    }
])

results = results.sort_values(
    "ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("")
print("")
print("============================================================")
print("EXPERIMENT 33 RESULTS")
print("============================================================")

print(
    results.to_string(index=False)
)

best_score = float(
    results.iloc[0]["ROC_AUC"]
)

print("")
print("23B / 31A benchmark: 0.945243")
print(
    f"Best Experiment 33: "
    f"{best_score:.6f}"
)

print(
    f"Difference vs 23B: "
    f"{best_score - 0.945243:+.6f}"
)

print("")
print("No submission generated.")
print("No prediction CSV generated.")
print("Experiment 33 complete.")

Training rows: 534932
Validation rows: 133733
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

33A - EXACT 23B CONTROL
Control feature count: 27

------------------------------------------------------------
33A_23B_Control
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 38)
33A_23B_Control ROC-AUC: 0.945243

Control score: 0.945243
Expected benchmark: 0.945243

33B - IDENTITY + DIGIT DECOMPOSITION
Digit attack feature count: 83

------------------------------------------------------------
33B_Identity_Frequency_Digits
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B_Identity_Frequency_Digits ROC-AUC: 0.945331


EXPERI